In [1]:
import os
os.chdir('/home/wiikai/factor')

import factorlab as lab
import pandas as pd
import numpy as np
import quool

industry_ret = quool.Factor("./data/industry-returns")

In [5]:
start = '20140101'
stop = '20240701'
name = '000300.XSHG'

shares = lab.quotes_day.read("circulation_a",  start=start,stop=stop)
price = lab.quotes_day.read("close",  start=start,stop=stop)
size = shares * price
ind = lab.industry_info.read('first_industry_name', start=start,stop=stop)
ret = price.pct_change(fill_method=None)

nonrealizable = lab.index_weights.read(name,start=start, stop=stop).isna()
ind = ind.where(~nonrealizable.astype(bool), other=np.nan).dropna(how='all',axis=1)
size = size.where(~nonrealizable.astype(bool), other=np.nan).dropna(how='all',axis=1)
ret = ret.where(~nonrealizable.astype(bool), other=np.nan).dropna(how='all',axis=1)

def get_weighted_industry_returns(market_cap, returns, industry):
    df = pd.DataFrame({
        'industry': industry.stack(),
        'market_cap': market_cap.stack(),
        'returns': returns.stack()
    })
    
    total_market_cap = df.groupby(['industry', df.index.get_level_values(0)])['market_cap'].sum()
    df['weighted_returns'] = df['market_cap'] * df['returns']
    weighted_returns = df.groupby(['industry', df.index.get_level_values(0)])['weighted_returns'].sum()
    res = (weighted_returns / total_market_cap).unstack(level=0)
    return res

def get_equal_weighted_industry_returns(returns, industry):
    df = pd.DataFrame({
        'industry': industry.stack(),
        'returns': returns.stack()
    })
    
    equal_weighted_returns = df.groupby(['industry', df.index.get_level_values(0)])['returns'].mean()
    res = equal_weighted_returns.unstack(level=0)
    return res

# data = get_size_weighted_industry_returns(size, ret, ind)
data = get_equal_weighted_industry_returns(ret, ind)
data

industry,交通运输,传媒,农林牧渔,医药,商贸零售,国防军工,基础化工,家电,建材,建筑,...,石油石化,纺织服装,综合,计算机,轻工制造,通信,钢铁,银行,非银行金融,食品饮料
date,,,,,,,,,,,,,,,,,,,,,
2014-01-02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
2014-01-03,-0.008385,0.003258,-0.016548,-0.011438,0.007650,-0.022792,-0.014278,-0.035057,-0.027342,-0.020517,...,-0.005961,-0.018405,-0.024197,0.008494,NaN,0.010519,0.005556,-0.017783,-0.029571,-0.009882
2014-01-06,-0.035237,-0.028567,-0.053635,-0.029265,-0.038460,-0.040444,-0.039801,-0.028968,-0.050505,-0.037757,...,0.002072,-0.055787,-0.035313,-0.042793,NaN,-0.014838,-0.025517,-0.015181,0.003344,-0.025584
2014-01-07,0.007928,0.010009,-0.004658,0.007526,0.010598,0.007515,0.006488,-0.001053,-0.009041,-0.005889,...,-0.003890,-0.001838,-0.005469,-0.000100,NaN,-0.012283,-0.010215,-0.004126,-0.008444,0.005484
2014-01-08,-0.014958,0.034377,-0.004277,0.006089,0.006733,0.019388,-0.010528,0.005573,-0.004260,-0.007570,...,-0.005870,-0.000626,0.013744,0.018199,NaN,0.005089,-0.011909,0.006081,0.006695,0.002763
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-06-25,0.001103,-0.013121,-0.004069,-0.006379,NaN,-0.013173,0.014085,0.007992,0.010050,-0.003195,...,0.007074,-0.007898,NaN,-0.021431,0.001072,-0.022618,0.000897,0.007628,-0.017360,0.001344
2024-06-26,0.002764,0.020681,-0.004637,0.009733,NaN,0.009810,-0.001396,-0.012940,0.010386,-0.001131,...,0.006585,-0.004286,NaN,0.048723,0.008978,0.029557,0.008550,0.001411,0.010369,0.001386
2024-06-27,-0.009030,-0.003862,-0.000197,-0.016757,NaN,-0.015576,-0.025012,-0.013904,-0.019617,0.002670,...,-0.008309,-0.025364,NaN,-0.015457,-0.022000,-0.015446,-0.010297,0.006630,-0.018959,-0.006157


In [7]:
data.index.name = 'date'
data = data.sort_index(ascending=False).unstack()
data.name = name

if name not in industry_ret.columns:
    industry_ret.add({name: data.dtype})
industry_ret.update(data)


In [8]:
industry_ret.read()

000985.XSHG  000300.XSHG
industry date                                
交通运输     2014-01-02     0.000000     0.000000
         2014-01-03    -0.016996    -0.008385
         2014-01-06    -0.032764    -0.035237
         2014-01-07     0.006232     0.007928
         2014-01-08    -0.012788    -0.014958
...                          ...          ...
通信       2024-07-01     0.014129     0.011273
钢铁       2024-07-01     0.017724     0.024523
银行       2024-07-01     0.015376     0.015363
非银行金融    2024-07-01    -0.000686    -0.003504
食品饮料     2024-07-01    -0.003395    -0.004352

[75118 rows x 2 columns]